# Timed-Release Cooperative Roles

Train and inspect the 8-ant shared-writes cooperative checkpoint with fixed release-rank timing.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib
import json
import re

import jax
from tqdm.auto import tqdm

from ant_byte_env.experiments import config_args_to_argv
from ant_byte_env.runs import write_json
from ant_byte_env.training.jax_mappo.timed_release import evaluation as timed_evaluation
from ant_byte_env.training.jax_mappo.timed_release import rendering as timed_rendering
from ant_byte_env.training.jax_mappo.timed_release import runner as timed_runner

workflows = importlib.reload(workflows)
timed_evaluation = importlib.reload(timed_evaluation)
timed_rendering = importlib.reload(timed_rendering)
timed_runner = importlib.reload(timed_runner)
print(f"JAX device: {jax.devices()[0]}")


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "timed_release_roles_8ants_shared_writes.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)
metadata = dict(experiment.metadata)

RUN_NAME = experiment.name
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
EVAL_DIR = RUN_DIR / "evaluation"
for directory in (CHECKPOINT_DIR, MEDIA_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Restore the source checkpoint first: {SOURCE_CHECKPOINT}")
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
BEST_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

CHUNK_PATTERN = re.compile(r"chunk_(\d+)_updates_(\d+)_(\d+)\.pkl$")


def existing_chunk_records():
    records = []
    for checkpoint_path in sorted(CHECKPOINT_DIR.glob("chunk_*_updates_*_*.pkl")):
        match = CHUNK_PATTERN.match(checkpoint_path.name)
        if not match:
            continue
        chunk_index, start_update, end_update = (int(value) for value in match.groups())
        records.append(
            {
                "chunk_index": chunk_index,
                "start_update": start_update,
                "end_update": end_update,
                "path": checkpoint_path,
            }
        )
    return records


UPDATE_TIMESTEPS = int(experiment_args["num_envs"]) * int(experiment_args["num_steps"])
BASE_TOTAL_UPDATES = int(experiment_args["total_timesteps"]) // UPDATE_TIMESTEPS
CHUNK_UPDATES = int(metadata.get("chunk_updates", 500))
CONTINUE_FROM_LATEST_CHUNK = bool(metadata.get("continue_from_latest_chunk", False))
CONTINUATION_UPDATES = int(metadata.get("continuation_updates", 0))
EVALUATION_EPISODES = int(metadata.get("evaluation_episodes", 4))
RENDER_ACTION_MODE = str(metadata.get("render_action_mode", "sampled_move_greedy_write"))
RENDER_MAX_FRAMES = int(metadata.get("render_max_frames", 480))
RENDER_TILE_SIZE = int(metadata.get("render_tile_size", workflows.NOTEBOOK_ROLLOUT_TILE_SIZE))

RUN_TRAINING = True
MAX_CHUNKS_TO_RUN = None
RESUME_FROM_BEST_CHECKPOINT = False
RUN_BEST_EVAL_DURING_TRAINING = bool(metadata.get("run_best_eval_during_training", False))

CHUNK_RECORDS = existing_chunk_records()
LATEST_CHUNK = max(CHUNK_RECORDS, key=lambda item: item["end_update"]) if CHUNK_RECORDS else None
START_COMPLETED_UPDATES = 0
START_CHUNK_INDEX = 0
ACTIVE_CHECKPOINT = SOURCE_CHECKPOINT
if CONTINUE_FROM_LATEST_CHUNK and LATEST_CHUNK is not None:
    START_COMPLETED_UPDATES = int(LATEST_CHUNK["end_update"])
    START_CHUNK_INDEX = int(LATEST_CHUNK["chunk_index"])
    ACTIVE_CHECKPOINT = LATEST_CHUNK["path"]
elif RESUME_FROM_BEST_CHECKPOINT and BEST_CHECKPOINT_PATH.exists():
    ACTIVE_CHECKPOINT = BEST_CHECKPOINT_PATH

if CONTINUE_FROM_LATEST_CHUNK and LATEST_CHUNK is not None:
    TOTAL_UPDATES = START_COMPLETED_UPDATES + CONTINUATION_UPDATES
else:
    TOTAL_UPDATES = BASE_TOTAL_UPDATES

summary = {
    "experiment": experiment.name,
    "source_checkpoint": str(SOURCE_CHECKPOINT),
    "run_dir": str(RUN_DIR),
    "base_total_updates": BASE_TOTAL_UPDATES,
    "target_total_updates": TOTAL_UPDATES,
    "start_completed_updates": START_COMPLETED_UPDATES,
    "continuation_updates": CONTINUATION_UPDATES,
    "continue_from_latest_chunk": CONTINUE_FROM_LATEST_CHUNK,
    "latest_chunk_checkpoint": str(LATEST_CHUNK["path"]) if LATEST_CHUNK else None,
    "chunk_updates": CHUNK_UPDATES,
    "release_interval": experiment_args["release_interval"],
    "initial_active_ants": experiment_args["initial_active_ants"],
    "max_steps": experiment_args["max_steps"],
    "num_envs": experiment_args["num_envs"],
    "num_steps": experiment_args["num_steps"],
    "critic_architecture": experiment_args["critic_architecture"],
    "actor_only_warm_start": experiment_args.get("actor_only_warm_start", False),
    "active_checkpoint": str(ACTIVE_CHECKPOINT),
    "training_rollout_temperature": experiment_args["training_rollout_temperature"],
    "eval_move_temperature": experiment_args["best_eval_move_temperature"],
    "run_best_eval_during_training": RUN_BEST_EVAL_DURING_TRAINING,
}
print(json.dumps(summary, indent=2))


In [ ]:
def progress_bar(label, total_updates):
    bar = tqdm(total=total_updates, desc=label)
    last = 0

    def callback(update, total, metrics):
        nonlocal last
        bar.total = total
        bar.update(max(0, int(update) - last))
        last = int(update)
        bar.set_postfix(
            ret=f"{metrics.get('episode_return', 0.0):.2f}",
            active=f"{metrics.get('mean_active_ants', 0.0):.2f}",
            delivered=f"{metrics.get('eval_mean_delivered_fraction', 0.0):.2f}",
        )

    return bar, callback


if RUN_TRAINING:
    previous_checkpoint = ACTIVE_CHECKPOINT
    chunk_metrics = []
    completed_updates = int(START_COMPLETED_UPDATES)
    chunk_index = int(START_CHUNK_INDEX)
    chunks_run = 0
    while completed_updates < TOTAL_UPDATES:
        if MAX_CHUNKS_TO_RUN is not None and chunks_run >= int(MAX_CHUNKS_TO_RUN):
            break
        updates = min(CHUNK_UPDATES, TOTAL_UPDATES - completed_updates)
        if updates <= 0:
            break
        chunk_index += 1
        label = f"chunk_{chunk_index:03d}_updates_{completed_updates:05d}_{completed_updates + updates:05d}"
        chunk_dir = RUN_DIR / label
        chunk_checkpoint = CHECKPOINT_DIR / f"{label}.pkl"
        chunk_args = dict(experiment_args)
        chunk_args["total_timesteps"] = updates * UPDATE_TIMESTEPS
        chunk_args["load_model"] = str(previous_checkpoint)
        chunk_args["save_model"] = str(chunk_checkpoint)
        if RUN_BEST_EVAL_DURING_TRAINING:
            chunk_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)
        else:
            chunk_args["save_best_model"] = None
            chunk_args["best_model_selection"] = "train"
        chunk_args["run_dir"] = str(chunk_dir)
        argv = config_args_to_argv(chunk_args)
        bar, callback = progress_bar(label, updates)
        try:
            metrics = timed_runner.main(argv, progress_callback=callback)
        finally:
            bar.close()
        chunk_metrics.append({"label": label, "checkpoint": str(chunk_checkpoint), **metrics})
        previous_checkpoint = (
            BEST_CHECKPOINT_PATH
            if RUN_BEST_EVAL_DURING_TRAINING and BEST_CHECKPOINT_PATH.exists()
            else chunk_checkpoint
        )
        completed_updates += updates
        chunks_run += 1
    ACTIVE_CHECKPOINT = previous_checkpoint
    write_json(
        RUN_DIR / "training_chunks.json",
        {
            "chunks": chunk_metrics,
            "active_checkpoint": str(ACTIVE_CHECKPOINT),
            "completed_updates": completed_updates,
            "target_total_updates": TOTAL_UPDATES,
            "continue_from_latest_chunk": CONTINUE_FROM_LATEST_CHUNK,
            "run_best_eval_during_training": RUN_BEST_EVAL_DURING_TRAINING,
        },
    )
else:
    chunk_metrics = []

ACTIVE_CHECKPOINT


In [ ]:
eval_metrics = timed_evaluation.evaluate_checkpoint(
    ACTIVE_CHECKPOINT,
    num_episodes=EVALUATION_EPISODES,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
eval_path = EVAL_DIR / f"timed_release_eval_{EVALUATION_EPISODES}ep.json"
write_json(eval_path, {"checkpoint": str(ACTIVE_CHECKPOINT), "metrics": eval_metrics})
eval_path, eval_metrics


In [ ]:
video_path = timed_rendering.render_timed_release_checkpoint(
    ACTIVE_CHECKPOINT,
    MEDIA_DIR / "timed_release_roles_rollout.mp4",
    max_frames=RENDER_MAX_FRAMES,
    tile_size=RENDER_TILE_SIZE,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
video_path
